# Single Caplet Deep Dive Testing

Detailed debugging and analysis of individual caplet pricing to diagnose arbitrage violations and forward rate bias issues.

## 5. Daily Instantaneous Forward Rates â€” Chart

Below we plot $f_{\text{key}}(0,t)$ and $f_{\text{ois}}(0,t)$ â€” the **instantaneous forward rates** at each day $t$ from today to 10Y.

**What these are:**
- $f(0,t) = dI(0,t)/dt$ â€” the rate applicable to an infinitesimal period around day $t$
- Computed from market period forwards: $F_{\text{period}}(T) \to P(0,T) \to I(0,T) = -\ln P \to f = dI/dt$
- The **key rate** $f_{\text{key}}$ is what the HW model simulates ($a_t$); the **OIS rate** $f_{\text{ois}}$ is used for discounting

The raw market forward points are overlaid â€” these are simple period forwards $F_{\text{period}}(T)$, which differ from $f(0,T)$ because of the compounding correction.

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import pickle
import sys
sys.path.insert(0, '../..')

from scipy.stats import norm
from pyquant.interest_rates import evaluate_timeline, build_fwd_curve, build_ifwd_curve_from_now_starting
from pyquant.torch_spline import PchipSpline1D

# Load market data
data_dir = Path('../../../../data')
vol_key_rate = pd.read_csv(data_dir / 'volatility_key_rate.csv')
fwd_ois = pd.read_csv(data_dir / 'forward_ois.csv')
fwd_key_rate = pd.read_csv(data_dir / 'forward_key_rate.csv')

# Build forward curve splines
ois_fwd_spline = build_fwd_curve(
    torch.tensor(fwd_ois['forward_rate'].values, dtype=torch.float32),
    torch.tensor(fwd_ois['time_to_maturity'].values, dtype=torch.float32)
)

key_fwd_spline = build_fwd_curve(
    torch.tensor(fwd_key_rate['forward_rate'].values, dtype=torch.float32),
    torch.tensor(fwd_key_rate['time_to_maturity'].values, dtype=torch.float32)
)

# Build integrated forward curves
ois_ifwd_curve = build_ifwd_curve_from_now_starting(
    torch.tensor(fwd_ois['forward_rate'].values, dtype=torch.float32),
    torch.tensor(fwd_ois['time_to_maturity'].values, dtype=torch.float32)
)

key_ifwd_curve = build_ifwd_curve_from_now_starting(
    torch.tensor(fwd_key_rate['forward_rate'].values, dtype=torch.float32),
    torch.tensor(fwd_key_rate['time_to_maturity'].values, dtype=torch.float32)
)

# Timeline setup
T_MAX = 10.0
N_STEPS = 3651
timeline = torch.linspace(0, T_MAX, N_STEPS + 1, dtype=torch.float32)
r0 = torch.tensor(fwd_ois['forward_rate'].iloc[0], dtype=torch.float32)
time_to_maturities = torch.tensor(sorted(vol_key_rate['time_to_maturity'].unique()), dtype=torch.float32)

print(f'Data loaded: {len(vol_key_rate)} caplets, {len(fwd_key_rate)} forward rates')
print(f'Timeline: {N_STEPS} steps over {T_MAX}Y')

Data loaded: 495 caplets, 14 forward rates
Timeline: 3651 steps over 10.0Y


## Load Calibrated Checkpoint

In [2]:
# Load calibrated model checkpoint
try:
    with open(data_dir / 'multi_theta_final.pkl', 'rb') as f:
        ckpt = pickle.load(f)
    print('Loaded V2 checkpoint:')
    print(f'  Iterations: {ckpt["iteration"]}')
    print(f'  Loss: {ckpt["loss"]:.6e}')
    print(f'  Theta range: [{ckpt["theta_values"].min():.6f}, {ckpt["theta_values"].max():.6f}]')
    print(f'  Epsilon: {ckpt["epsilon"]:.6f}')
except FileNotFoundError:
    print('No V2 checkpoint found. Run calibration first.')
    ckpt = None

Loaded V2 checkpoint:
  Iterations: 53
  Loss: 1.781081e-02
  Theta range: [0.000648, 0.004425]
  Epsilon: 0.499261


## CIR and Model Generators

Same functions as in main notebook for standalone testing.

In [3]:
def noncentral_chisquare(df: torch.Tensor, nonc: torch.Tensor) -> torch.Tensor:
    PSI_CRIT = 1.5
    m = df + nonc
    s2 = 2*df + 4*nonc
    psi = s2 / (m**2 + 1e-12)
    psi_inv = 1 / (psi + 1e-12)
    b2 = 2*psi_inv - 1 + torch.sqrt(2*psi_inv) * torch.sqrt(2*psi_inv - 1 + 1e-12)
    a = m / (1 + b2)
    sample_quad = a * (torch.sqrt(b2) + torch.randn_like(a))**2
    p = (psi - 1) / (psi + 1)
    beta = (1 - p) / (m + 1e-12)
    u = torch.rand_like(p)
    sample_exp = torch.where(
        u > p,
        torch.log((1 - p) / (1 - u + 1e-12)) / (beta + 1e-12),
        torch.zeros_like(u)
    )
    return torch.where(psi <= PSI_CRIT, sample_quad, sample_exp)

def generate_cir_custom(n_paths, timeline, v0, kappa, theta_t, eps_t, minimum_value=1e-9):
    device = timeline.device
    dt = timeline.diff()
    n_steps = len(dt)
    v_paths = torch.zeros((n_paths, n_steps + 1), dtype=torch.float32, device=device)
    v_paths[:, 0] = v0
    if eps_t.dim() == 0:
        eps_vec = eps_t.expand(n_steps + 1)
    else:
        eps_vec = eps_t
    for i in range(n_steps):
        dt_i = dt[i]
        theta_i = theta_t[i]
        eps_i = eps_vec[i]
        v_curr = v_paths[:, i]
        exp_term = torch.exp(-kappa * dt_i)
        one_minus_exp = 1 - exp_term
        denom = eps_i**2 * (one_minus_exp + 1e-12)
        c_bar = one_minus_exp / (4 * kappa + 1e-12) * eps_i**2
        kappa_bar = 4 * kappa * exp_term * v_curr / denom
        delta = 4 * kappa * theta_i / (eps_i**2 + 1e-12)
        v_next = c_bar * noncentral_chisquare(delta, kappa_bar)
        if minimum_value is not None:
            v_next = torch.clamp(v_next, min=minimum_value)
        v_paths[:, i+1] = v_next
    return v_paths

def generate_hw_sv(n_paths, timeline, x0, lam, v_paths):
    device = timeline.device
    dt = timeline.diff()
    n_steps = len(dt)
    x_paths = torch.zeros((n_paths, n_steps + 1), dtype=torch.float32, device=device)
    x_paths[:, 0] = x0
    for i in range(n_steps):
        x = x_paths[:, i]
        v = v_paths[:, i]
        dW = torch.randn(n_paths, device=device) * torch.sqrt(dt[i])
        x_next = x * torch.exp(-lam * dt[i]) + torch.sqrt(torch.clamp(v, min=1e-9)) * dW
        x_paths[:, i + 1] = x_next
    return x_paths

def generate_ou_spread(n_paths, timeline, k0, gamma, xi):
    device = timeline.device
    dt = timeline.diff()
    n_steps = len(dt)
    k_paths = torch.zeros((n_paths, n_steps + 1), dtype=torch.float32, device=device)
    k_paths[:, 0] = k0
    for i in range(n_steps):
        k = k_paths[:, i]
        dW = torch.randn(n_paths, device=device) * torch.sqrt(dt[i])
        k_next = k * torch.exp(-gamma * dt[i]) + xi * dW
        k_paths[:, i + 1] = k_next
    return k_paths

def create_key_rate_model(timeline, n_paths, key_fwd_spline, ois_fwd_spline, r0, a0, 
                           v0, kappa, theta_t, eps_t, x0, lam, k0, gamma, xi):
    eps_val = eps_t if eps_t.dim() > 0 else eps_t
    v_paths = generate_cir_custom(n_paths=n_paths, timeline=timeline, v0=v0, kappa=kappa,
                                   theta_t=theta_t, eps_t=eps_val, minimum_value=1e-9)
    x_paths = generate_hw_sv(n_paths, timeline, x0, lam, v_paths)
    k_paths = generate_ou_spread(n_paths, timeline, k0, gamma, xi)
    if isinstance(ois_fwd_spline, PchipSpline1D):
        f_ois_t = ois_fwd_spline.evaluate(timeline)
    else:
        f_ois_t = ois_fwd_spline.mean()
    r_paths = f_ois_t.unsqueeze(0) + x_paths if f_ois_t.dim() == 1 else ois_fwd_spline.mean() + x_paths
    if isinstance(key_fwd_spline, PchipSpline1D):
        f_key_t = key_fwd_spline.evaluate(timeline)
        a_paths = f_key_t.unsqueeze(0) + x_paths + k_paths
    else:
        spread = key_fwd_spline.mean() - (f_ois_t.mean() if f_ois_t.dim() == 1 else ois_fwd_spline.mean())
        a_paths = r_paths + spread + k_paths
    return {'ois_rate_paths': r_paths, 'key_rate_paths': a_paths, 'variance_paths': v_paths, 'timeline': timeline, 'n_paths': n_paths}

## Forward Rate Bias Diagnostic

Check if simulated forwards match market forwards at multiple maturities.

In [11]:
# Check forward rate bias across multiple maturities
print("="*80)
print("FORWARD RATE BIAS DIAGNOSTIC")
print("="*80)

n_paths_high = 5000
print(f"\nUsing {n_paths_high} MC paths\n")

if ckpt is not None:
    theta_t_high = PchipSpline1D(ckpt['time_to_maturities'], ckpt['theta_values']).evaluate(timeline)
    eps_t_high = ckpt['epsilon'] * torch.ones_like(timeline)

    # BUG FIX: Use key_fwd_spline and ois_fwd_spline (instantaneous forward curves)
    # NOT key_ifwd_curve and ois_ifwd_curve (integrated forward curves)!
    model_high = create_key_rate_model(
        timeline, n_paths_high, key_fwd_spline, ois_fwd_spline,
        r0, r0, ckpt['v0'], ckpt['kappa'], theta_t_high, eps_t_high,
        torch.tensor(0.0), ckpt['lam'], torch.tensor(0.0), ckpt['gamma'], ckpt['xi']
    )

    a_paths_high = model_high['key_rate_paths']

    test_maturities = [1.0, 2.0, 3.0, 5.0, 7.0, 10.0]
    print(f"{'Maturity':<10} {'Market F':<12} {'Simulated L':<15} {'Bias (bps)':<12} {'Ratio'}")
    print("="*70)

    for T in test_maturities:
        if T > T_MAX:
            continue
        market_fwd = key_fwd_spline.evaluate(torch.tensor([T])).item()
        idx_start = torch.argmin(torch.abs(timeline - T))
        idx_end = torch.argmin(torch.abs(timeline - (T + 0.25)))
        a_accrual = a_paths_high[:, idx_start:idx_end+1]
        dt = timeline[idx_start:idx_end+1].diff()
        L_sim = (a_accrual[:, :-1] * dt.unsqueeze(0)).sum(dim=1) / dt.sum()
        L_mean = L_sim.mean().item()
        bias_bps = (L_mean - market_fwd) * 10000
        ratio = L_mean / market_fwd
        print(f"{T:<10.1f} {market_fwd*100:<12.2f}% {L_mean*100:<15.2f}% {bias_bps:<12.0f} {ratio:<.2f}x")

    print("\n" + "="*80)
    print("Under risk-neutral measure, E[L(T)] should equal F(T)")
    print("="*80)
else:
    print("No checkpoint loaded - run calibration first")

FORWARD RATE BIAS DIAGNOSTIC

Using 5000 MC paths

Maturity   Market F     Simulated L     Bias (bps)   Ratio
1.0        15.16       % 14.93          % -23          0.99x
2.0        13.52       % 13.43          % -9           0.99x
3.0        12.93       % 12.87          % -6           1.00x
5.0        12.40       % 12.48          % 8            1.01x
7.0        12.14       % 12.22          % 8            1.01x
10.0       11.94       % nan            % nan          nanx

Under risk-neutral measure, E[L(T)] should equal F(T)


## Single Caplet Deep Dive

Trace through one specific caplet calculation step-by-step.

In [5]:
# Single caplet analysis
if ckpt is not None:
    print("="*80)
    print("SINGLE CAPLET ANALYSIS")
    print("="*80)

    # Select a caplet to analyze
    T_fix = 3.0  # 3Y maturity
    K = 0.13  # 13% strike (close to ATM)
    tau = 0.25  # Quarterly accrual

    # Get market forward
    F = key_fwd_spline.evaluate(torch.tensor([T_fix])).item()
    
    # Get market vol for this caplet
    caplet_row = vol_key_rate[(vol_key_rate['time_to_maturity'] == T_fix) & 
                              (abs(vol_key_rate['strike'] - K) < 0.01)]
    if len(caplet_row) > 0:
        sigma_mkt = caplet_row['implied_normal_vol'].values[0]
    else:
        sigma_mkt = vol_key_rate['implied_normal_vol'].mean()

    print(f"\nSelected Caplet:")
    print(f"  Maturity (T_fix): {T_fix:.2f}Y")
    print(f"  Strike (K):       {K*100:.2f}%")
    print(f"  Forward (F):      {F*100:.2f}%")
    print(f"  Market Vol (Ïƒ):   {sigma_mkt*100:.2f}%")

    # Simulate with high paths
    n_paths_debug = 5000
    theta_t_debug = evaluate_timeline(timeline, ckpt['time_to_maturities'], ckpt['theta_values'])
    eps_t_debug = ckpt['epsilon'] * torch.ones_like(timeline)

    with torch.no_grad():
        model_debug = create_key_rate_model(
            timeline, n_paths_debug, key_fwd_spline, ois_fwd_spline, r0, r0,
            ckpt['v0'], ckpt['kappa'], theta_t_debug, eps_t_debug,
            torch.tensor(0.0), ckpt['lam'], torch.tensor(0.0), ckpt['gamma'], ckpt['xi']
        )

        a_paths = model_debug['key_rate_paths']
        r_paths = model_debug['ois_rate_paths']
        v_paths = model_debug['variance_paths']

        idx_fix_start = torch.argmin(torch.abs(timeline - T_fix))
        idx_fix_end = torch.argmin(torch.abs(timeline - (T_fix + tau)))

        # Realized forward
        if idx_fix_end > idx_fix_start:
            a_accrual = a_paths[:, idx_fix_start:idx_fix_end+1]
            dt_accrual = timeline[idx_fix_start:idx_fix_end+1].diff()
            L_realized = (a_accrual[:, :-1] * dt_accrual.unsqueeze(0)).sum(dim=1) / dt_accrual.sum()
        else:
            L_realized = a_paths[:, idx_fix_start]

        # Discount factor
        r_discount = r_paths[:, :idx_fix_end+1]
        dt_discount = timeline[:idx_fix_end+1].diff()
        integral_r = (r_discount[:, :-1] * dt_discount.unsqueeze(0)).sum(dim=1)
        P_disc_paths = torch.exp(-integral_r)

        # Payoff
        payoff_per_path = torch.clamp(L_realized - K, min=0) * tau * P_disc_paths
        pv_model = payoff_per_path.mean()

        # Intrinsic
        intrinsic = max(F - K, 0) * tau * P_disc_paths.mean().item()

        print(f"\n" + "-"*80)
        print(f"PATH STATISTICS ({n_paths_debug} paths)")
        print(f"-"*80)
        print(f"\nRealized Forward L:")
        print(f"  Mean:   {L_realized.mean()*100:.4f}%")
        print(f"  Std:    {L_realized.std()*100:.4f}%")
        print(f"  Market: {F*100:.4f}%")
        print(f"  Bias:   {(L_realized.mean()-F)*10000:.1f} bps")

        print(f"\nDiscount Factor P(0, T+Ï„):")
        print(f"  Mean:   {P_disc_paths.mean():.6f}")

        print(f"\nCaplet Pricing:")
        print(f"  Model PV:     {pv_model:.6e}")
        print(f"  Intrinsic:    {intrinsic:.6e}")
        print(f"  Ratio:        {pv_model/intrinsic if intrinsic > 0 else 0:.4f}")
        print(f"  ITM paths:    {(payoff_per_path > 0).sum()}/{n_paths_debug} ({(payoff_per_path > 0).sum()/n_paths_debug*100:.1f}%)")

        if pv_model / intrinsic < 0.99 and intrinsic > 0:
            print(f"\nâŒ ARBITRAGE VIOLATION: Model PV < Intrinsic!")
        else:
            print(f"\nâœ… No arbitrage violation")

    print("="*80)
else:
    print("No checkpoint loaded - run calibration first")

SINGLE CAPLET ANALYSIS

Selected Caplet:
  Maturity (T_fix): 3.00Y
  Strike (K):       13.00%
  Forward (F):      12.93%
  Market Vol (Ïƒ):   3.13%

--------------------------------------------------------------------------------
PATH STATISTICS (5000 paths)
--------------------------------------------------------------------------------

Realized Forward L:
  Mean:   12.9420%
  Std:    3.8751%
  Market: 12.9300%
  Bias:   1.2 bps

Discount Factor P(0, T+Ï„):
  Mean:   0.629170

Caplet Pricing:
  Model PV:     9.620336e-04
  Intrinsic:    0.000000e+00
  Ratio:        0.0000
  ITM paths:    2267/5000 (45.3%)

âœ… No arbitrage violation


## Initial Conditions Check

In [6]:
# Check initial conditions and forward curve at t=0
print("="*80)
print("INITIAL CONDITIONS CHECK")
print("="*80)

f_key_0 = key_fwd_spline.evaluate(torch.tensor([0.0])).item()
f_key_1 = key_fwd_spline.evaluate(torch.tensor([1.0])).item()
f_ois_0 = ois_fwd_spline.evaluate(torch.tensor([0.0])).item()

print(f"\nForward Curve Values:")
print(f"  f_KEY(0):  {f_key_0*100:.4f}%")
print(f"  f_KEY(1):  {f_key_1*100:.4f}%")
print(f"  f_OIS(0):  {f_ois_0*100:.4f}%")

print(f"\nExpected:")
print(f"  f_KEY(0) should match first data point: {fwd_key_rate['forward_rate'].iloc[0]*100:.2f}%")
print(f"  f_OIS(0) should match first data point: {fwd_ois['forward_rate'].iloc[0]*100:.2f}%")

if abs(f_key_0 - fwd_key_rate['forward_rate'].iloc[0]) < 0.001:
    print("\nâœ… Forward curves properly anchored at t=0")
else:
    print("\nâŒ Forward curve anchoring issue detected!")

print("="*80)

INITIAL CONDITIONS CHECK

Forward Curve Values:
  f_KEY(0):  16.0000%
  f_KEY(1):  15.1600%
  f_OIS(0):  15.2700%

Expected:
  f_KEY(0) should match first data point: 16.00%
  f_OIS(0) should match first data point: 15.27%

âœ… Forward curves properly anchored at t=0


## Single Caplet Verification Test

Comprehensive step-by-step verification of:
1. Bachelier pricing formula
2. Intrinsic value constraints
3. Monte Carlo simulation correctness
4. Calendar arbitrage detection
5. Butterfly arbitrage detection
6. Batch pricing consistency

In [8]:
# ============================================================================
# SINGLE CAPLET VERIFICATION TEST
# ============================================================================
# Test all components step-by-step to verify correctness

print("="*80)
print("SINGLE CAPLET VERIFICATION TEST")
print("="*80)

# Setup device and parameters
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tau = 0.25
dt = timeline[1] - timeline[0]

# Pre-compute forward curves on timeline
f_key_vec = key_fwd_spline.evaluate(timeline).to(device)
f_ois_vec = ois_fwd_spline.evaluate(timeline).to(device)
timeline_device = timeline.to(device)

# Pre-compute caplet data
caplet_data = vol_key_rate[['time_to_maturity', 'strike', 'implied_normal_vol']].values
T_fixes = torch.tensor(caplet_data[:, 0], dtype=torch.float32, device=device)
strikes = torch.tensor(caplet_data[:, 1], dtype=torch.float32, device=device)
market_vols = torch.tensor(caplet_data[:, 2], dtype=torch.float32, device=device)

# Pre-compute timeline indices
idx_fixes = torch.searchsorted(timeline_device, T_fixes).clamp(0, len(timeline_device)-1)
idx_pays = torch.searchsorted(timeline_device, T_fixes + tau).clamp(0, len(timeline_device)-1)
market_fwds = key_fwd_spline.evaluate(T_fixes.cpu()).to(device)

# Pick a single caplet for detailed testing
test_idx = 100  # Mid-range caplet
T_test = T_fixes[test_idx].item()
K_test = strikes[test_idx].item()
vol_test = market_vols[test_idx].item()
F_test = market_fwds[test_idx].item()

print(f"\n1. CAPLET SPECIFICATION:")
print(f"   Maturity (T_fix): {T_test:.2f} years")
print(f"   Strike (K):       {K_test*100:.2f}%")
print(f"   Forward (F):      {F_test*100:.2f}%")
print(f"   Market Vol (Ïƒ):   {vol_test*100:.2f}%")
print(f"   Tau (accrual):    {tau} years")
print(f"   Moneyness (F-K):  {(F_test - K_test)*100:.2f}% ({'ITM' if F_test > K_test else 'OTM'})")

# ============================================================================
# 2. BACHELIER PRICE CALCULATION (MARKET)
# ============================================================================
print(f"\n2. BACHELIER MARKET PRICE:")

T_pay_test = T_test + tau
sqrt_T_test = np.sqrt(T_test)

# Discount factor from OIS curve
idx_pay_test = int(T_pay_test / dt.item())
idx_pay_test = min(idx_pay_test, len(f_ois_vec) - 1)
disc_test = torch.exp(-f_ois_vec[:idx_pay_test+1].sum() * dt).item()

print(f"   Payment date:     {T_pay_test:.2f} years")
print(f"   Discount factor:  {disc_test:.6f}")
print(f"   sqrt(T):          {sqrt_T_test:.4f}")

# Bachelier d
d_test = (F_test - K_test) / (vol_test * sqrt_T_test + 1e-10)
print(f"   d = (F-K)/(ÏƒâˆšT):  {d_test:.4f}")

# Bachelier undiscounted price
undiscounted_pv_test = (F_test - K_test) * norm.cdf(d_test) + vol_test * sqrt_T_test * norm.pdf(d_test)
market_pv_test = tau * disc_test * undiscounted_pv_test
print(f"   Undiscounted PV:  {undiscounted_pv_test*10000:.4f} bp")
print(f"   Market PV:        {market_pv_test*10000:.4f} bp")

# ============================================================================
# 3. INTRINSIC VALUE CHECK
# ============================================================================
print(f"\n3. INTRINSIC VALUE CHECK:")
intrinsic_test = tau * disc_test * max(0, F_test - K_test)
print(f"   Intrinsic = Ï„Â·DÂ·max(0, F-K)")
print(f"   Intrinsic value:  {intrinsic_test*10000:.4f} bp")
print(f"   Market PV:        {market_pv_test*10000:.4f} bp")
print(f"   Time value:       {(market_pv_test - intrinsic_test)*10000:.4f} bp")
print(f"   PV >= Intrinsic:  {'âœ“' if market_pv_test >= intrinsic_test - 1e-10 else 'âœ— ARBITRAGE!'}")

# ============================================================================
# 4. SUMMARY
# ============================================================================
print(f"\n{'='*80}")
print("BACHELIER CHECK SUMMARY")
print("="*80)
market_intrinsic_ok = market_pv_test >= intrinsic_test - 1e-10
print(f"  âœ“ Intrinsic (Market):    {'PASS' if market_intrinsic_ok else 'FAIL'}")
print("="*80)

SINGLE CAPLET VERIFICATION TEST

1. CAPLET SPECIFICATION:
   Maturity (T_fix): 0.50 years
   Strike (K):       4.00%
   Forward (F):      16.33%
   Market Vol (Ïƒ):   3.10%
   Tau (accrual):    0.25 years
   Moneyness (F-K):  12.33% (ITM)

2. BACHELIER MARKET PRICE:
   Payment date:     0.75 years
   Discount factor:  0.886777
   sqrt(T):          0.7071
   d = (F-K)/(ÏƒâˆšT):  5.6198
   Undiscounted PV:  1232.9999 bp
   Market PV:        273.3490 bp

3. INTRINSIC VALUE CHECK:
   Intrinsic = Ï„Â·DÂ·max(0, F-K)
   Intrinsic value:  273.3490 bp
   Market PV:        273.3490 bp
   Time value:       0.0000 bp
   PV >= Intrinsic:  âœ“

BACHELIER CHECK SUMMARY
  âœ“ Intrinsic (Market):    PASS


In [ ]:
# ============================================================================
# CALENDAR AND BUTTERFLY ARBITRAGE CHECK
# ============================================================================
print("="*80)
print("CALENDAR AND BUTTERFLY ARBITRAGE CHECK")
print("="*80)

# Use same test caplet from previous cell
print(f"\nTest caplet: T={T_test:.2f}Y, K={K_test*100:.2f}%, F={F_test*100:.2f}%")

# ============================================================================
# CALENDAR ARBITRAGE: dw/dT >= 0
# ============================================================================
print(f"\n1. CALENDAR ARBITRAGE CHECK (dw/dT >= 0):")
print("   Total variance w = ÏƒÂ²T must increase with maturity at fixed strike")

# Find caplets at same strike, different maturities
same_K_mask = torch.abs(strikes - K_test) < 1e-6
T_at_K = T_fixes[same_K_mask].cpu().numpy()
vol_at_K = market_vols[same_K_mask].cpu().numpy()

# Sort by maturity
sort_idx = np.argsort(T_at_K)
T_sorted = T_at_K[sort_idx]
vol_sorted = vol_at_K[sort_idx]
w_sorted = vol_sorted**2 * T_sorted  # Total variance

print(f"   Strike = {K_test*100:.2f}%")
print(f"   Maturities found: {len(T_sorted)}")
print(f"\n   T(Y)    Vol(%)   w=ÏƒÂ²T     dw/dT")
print(f"   {'='*45}")

calendar_violations = 0
for i in range(len(T_sorted)):
    if i == 0:
        dw_dT = np.nan
    else:
        dw_dT = (w_sorted[i] - w_sorted[i-1]) / (T_sorted[i] - T_sorted[i-1])
    
    violation = '' if np.isnan(dw_dT) or dw_dT >= -1e-6 else ' â† VIOLATION!'
    if 'VIOLATION' in violation:
        calendar_violations += 1
    dw_str = f'{dw_dT:.6f}' if not np.isnan(dw_dT) else 'N/A'
    print(f"   {T_sorted[i]:5.2f}   {vol_sorted[i]*100:6.3f}   {w_sorted[i]:.6f}  {dw_str:>10}{violation}")

print(f"\n   Calendar violations at this strike: {calendar_violations}")

# ============================================================================
# BUTTERFLY ARBITRAGE: dÂ²C/dKÂ² >= 0
# ============================================================================
print(f"\n2. BUTTERFLY ARBITRAGE CHECK (dÂ²C/dKÂ² >= 0):")
print("   Caplet price must be convex in strike (no negative butterflies)")

# Find caplets at same maturity, different strikes
same_T_mask = torch.abs(T_fixes - T_test) < 1e-6
K_at_T = strikes[same_T_mask].cpu().numpy()
vol_at_T = market_vols[same_T_mask].cpu().numpy()

# Sort by strike
sort_idx = np.argsort(K_at_T)
K_sorted = K_at_T[sort_idx]
vol_sorted_K = vol_at_T[sort_idx]

# Get forward for this maturity
F_at_T = F_test
sqrt_T = np.sqrt(T_test)

# Compute Bachelier prices at each strike
C_sorted = []
for K_i, vol_i in zip(K_sorted, vol_sorted_K):
    d_i = (F_at_T - K_i) / (vol_i * sqrt_T + 1e-10)
    C_i = tau * disc_test * ((F_at_T - K_i) * norm.cdf(d_i) + vol_i * sqrt_T * norm.pdf(d_i))
    C_sorted.append(C_i)
C_sorted = np.array(C_sorted)

print(f"   Maturity = {T_test:.2f}Y, Forward = {F_at_T*100:.2f}%")
print(f"   Strikes found: {len(K_sorted)}")
print(f"\n   K(%)    Vol(%)   C(bp)    dÂ²C/dKÂ²")
print(f"   {'='*50}")

butterfly_violations = 0
for i in range(len(K_sorted)):
    if i == 0 or i == len(K_sorted) - 1:
        d2C = np.nan
    else:
        h = (K_sorted[i+1] - K_sorted[i-1]) / 2
        d2C = (C_sorted[i+1] - 2*C_sorted[i] + C_sorted[i-1]) / (h**2)
    
    violation = '' if np.isnan(d2C) or d2C >= -1e-8 else ' â† VIOLATION!'
    if 'VIOLATION' in violation:
        butterfly_violations += 1
    d2C_str = f'{d2C:.4f}' if not np.isnan(d2C) else 'N/A'
    print(f"   {K_sorted[i]*100:5.2f}   {vol_sorted_K[i]*100:6.3f}   {C_sorted[i]*10000:7.2f}  {d2C_str:>10}{violation}")

print(f"\n   Butterfly violations at this maturity: {butterfly_violations}")

# ============================================================================
# SUMMARY
# ============================================================================
print(f"\n{'='*80}")
print("ARBITRAGE CHECK SUMMARY")
print("="*80)
print(f"  Calendar violations (dw/dT < 0):    {calendar_violations}")
print(f"  Butterfly violations (dÂ²C/dKÂ² < 0): {butterfly_violations}")
print(f"  Overall arbitrage-free:             {'YES' if calendar_violations == 0 and butterfly_violations == 0 else 'NO'}")
print("="*80)

In [ ]:
# ============================================================================
# ARBITRAGE FUNCTION VERIFICATION WITH KNOWN TEST CASES
# ============================================================================
print("="*80)
print("ARBITRAGE FUNCTION VERIFICATION")
print("="*80)

import importlib
import affine_calibration.caplet_vol_surface as cvs
importlib.reload(cvs)
from affine_calibration.caplet_vol_surface import check_market_arbitrage

# Test 1: FLAT VOL SURFACE (should be arbitrage-free)
print("\n1. TEST WITH FLAT VOL SURFACE (should be arbitrage-free):")

# Create synthetic flat vol surface
T_grid = np.array([1.0, 2.0, 3.0, 5.0, 7.0, 10.0])
K_grid = np.array([0.05, 0.10, 0.15, 0.20, 0.25])
flat_vol = 0.05  # 5% flat vol

flat_vol_df = pd.DataFrame([
    {'time_to_maturity': T, 'strike': K, 'implied_normal_vol': flat_vol}
    for T in T_grid for K in K_grid
])

print(f"   Created {len(flat_vol_df)} points with flat vol = {flat_vol*100:.1f}%")

# Check total variance (should be increasing in T)
print(f"   Total variance w = ÏƒÂ²T at each maturity:")
for K in K_grid[:2]:  # Just check first 2 strikes
    mask = flat_vol_df['strike'] == K
    T_vals = flat_vol_df.loc[mask, 'time_to_maturity'].values
    w_vals = flat_vol**2 * T_vals  # Total variance
    print(f"     K={K*100:.0f}%: T={T_vals}, w={w_vals}")
    if len(w_vals) > 1:
        dw_dT = np.diff(w_vals) / np.diff(T_vals)
        print(f"       dw/dT = {dw_dT} (all should be > 0)")

# Run arbitrage check
try:
    flat_arb_df, flat_arb_summary = check_market_arbitrage(
        flat_vol_df, fwd_key_rate, fwd_ois,
        tau=0.25, h_T=0.1, h_K=0.005,
        plot=False, verbose=False,
        surface_name="FlatVol"
    )
    print(f"\n   Results:")
    print(f"   Calendar violations: {flat_arb_summary['calendar_violations']}")
    print(f"   Butterfly violations: {flat_arb_summary['butterfly_violations']}")
    print(f"   Arbitrage-free: {flat_arb_summary['is_arbitrage_free']}")
    test1_pass = flat_arb_summary['calendar_violations'] == 0
    print(f"   TEST 1: {'PASS' if test1_pass else 'FAIL'}")
except Exception as e:
    print(f"   ERROR: {e}")
    test1_pass = False

# Test 2: CALENDAR ARBITRAGE (decreasing total variance)
print("\n" + "-"*80)
print("\n2. TEST WITH CALENDAR ARBITRAGE (decreasing total variance):")

# Create surface with decreasing w = ÏƒÂ²T (calendar arbitrage)
cal_arb_df = pd.DataFrame([
    # At K=10%, vol DECREASES faster than 1/sqrt(T) -> w decreases
    {'time_to_maturity': 1.0, 'strike': 0.10, 'implied_normal_vol': 0.10},  # w = 0.01
    {'time_to_maturity': 2.0, 'strike': 0.10, 'implied_normal_vol': 0.05},  # w = 0.005 < 0.01 ARBITRAGE!
    {'time_to_maturity': 3.0, 'strike': 0.10, 'implied_normal_vol': 0.04},  # w = 0.0048
])

print(f"   Created surface with DECREASING total variance:")
print(f"   T=1Y: vol=10%, w=ÏƒÂ²T = {0.10**2 * 1:.4f}")
print(f"   T=2Y: vol=5%,  w=ÏƒÂ²T = {0.05**2 * 2:.4f} <- LESS than T=1Y!")
print(f"   T=3Y: vol=4%,  w=ÏƒÂ²T = {0.04**2 * 3:.4f}")

try:
    cal_arb_result, cal_arb_summary = check_market_arbitrage(
        cal_arb_df, fwd_key_rate, fwd_ois,
        tau=0.25, h_T=0.1, h_K=0.005,
        plot=False, verbose=False,
        surface_name="CalArb"
    )
    print(f"\n   Results:")
    print(f"   Calendar violations detected: {cal_arb_summary['calendar_violations']}")
    print(f"   Expected: >= 1 violation")
    test2_pass = cal_arb_summary['calendar_violations'] >= 1
    print(f"   TEST 2: {'PASS' if test2_pass else 'FAIL - should detect violation!'}")
except Exception as e:
    print(f"   ERROR: {e}")
    test2_pass = False

# Test 3: ACTUAL MARKET DATA
print("\n" + "-"*80)
print("\n3. ACTUAL MARKET DATA:")

try:
    mkt_arb_df, mkt_arb_summary = check_market_arbitrage(
        vol_key_rate, fwd_key_rate, fwd_ois,
        tau=0.25, h_T=0.1, h_K=0.005,
        plot=False, verbose=True,
        surface_name="Market"
    )
    test3_run = True
except Exception as e:
    print(f"   ERROR: {e}")
    test3_run = False

# Summary
print("\n" + "="*80)
print("ARBITRAGE FUNCTION TEST SUMMARY")
print("="*80)
print(f"  Test 1 (Flat vol = arb-free):     {'PASS' if test1_pass else 'FAIL'}")
print(f"  Test 2 (Calendar arb detected):   {'PASS' if test2_pass else 'FAIL'}")
print(f"  Test 3 (Market data runs):        {'PASS' if test3_run else 'FAIL'}")
all_pass = test1_pass and test2_pass and test3_run
print(f"  Overall:                          {'ALL TESTS PASSED' if all_pass else 'SOME TESTS FAILED'}")
print("="*80)

## Intrinsic Value Diagnostic

Check all caplets for intrinsic value violations (model PV < intrinsic) and analyze forward rate bias.

In [10]:
# ============================================================================
# INTRINSIC VALUE DIAGNOSTIC - WHY MODEL < INTRINSIC?
# ============================================================================
print("="*80)
print("INTRINSIC VALUE DIAGNOSTIC")
print("="*80)

# Check if checkpoint was loaded
if ckpt is None:
    print("No checkpoint loaded - skipping diagnostic. Run calibration first.")
else:
    print(f"Checkpoint keys: {list(ckpt.keys())}")
    
    # Setup simulation parameters from checkpoint
    theta_diag = ckpt['theta_values'].to(device)
    # Use 'time_to_maturities' instead of 'theta_nodes' if available
    theta_nodes_diag = ckpt.get('theta_nodes', ckpt.get('time_to_maturities', time_to_maturities)).to(device)
    eps_diag = ckpt.get('epsilon', 0.15)
    v0_diag = ckpt.get('v0', theta_diag[0].item())
    kappa_diag = ckpt.get('kappa', 3.0)
    lam_diag = ckpt.get('lam', 0.3)
    gamma_diag = ckpt.get('gamma', 0.3)
    xi_diag = ckpt.get('xi', 0.01)
    
    print(f"Theta nodes: {theta_nodes_diag.cpu().numpy()}")
    print(f"Theta values: {theta_diag.cpu().numpy()}")
    print(f"Epsilon: {eps_diag}, v0: {v0_diag}")
    
    # Need theta_to_vec and simulation functions - use same code from fast_calibration
    def theta_to_vec(theta_vals, theta_nodes, timeline):
        """Interpolate theta values to full timeline using PCHIP."""
        spline = PchipSpline1D(theta_nodes.cpu(), theta_vals.cpu())
        return spline.evaluate(timeline.cpu()).to(timeline.device)
    
    def fast_cir_paths(n_paths, n_steps, dt, v0, kappa, theta_vec, epsilon, device, randn=None):
        v = torch.zeros((n_paths, n_steps + 1), dtype=torch.float32, device=device)
        v[:, 0] = v0
        sqrt_dt = np.sqrt(dt)
        if randn is None:
            randn = torch.randn(n_paths, n_steps, device=device)
        for i in range(n_steps):
            v_curr = v[:, i]
            theta_i = theta_vec[i] if theta_vec.dim() > 0 else theta_vec
            drift = kappa * (theta_i - v_curr) * dt
            diffusion = epsilon * torch.sqrt(torch.clamp(v_curr, min=1e-8)) * sqrt_dt * randn[:, i]
            v[:, i+1] = torch.abs(v_curr + drift + diffusion)
        return v
    
    def fast_ou_paths(n_paths, n_steps, dt, x0, lam, vol_paths, device, randn=None):
        x = torch.zeros((n_paths, n_steps + 1), dtype=torch.float32, device=device)
        x[:, 0] = x0
        sqrt_dt = np.sqrt(dt)
        exp_lam_dt = np.exp(-lam * dt)
        if randn is None:
            randn = torch.randn(n_paths, n_steps, device=device)
        for i in range(n_steps):
            x[:, i+1] = x[:, i] * exp_lam_dt + torch.sqrt(torch.clamp(vol_paths[:, i], min=1e-9)) * sqrt_dt * randn[:, i]
        return x
    
    def fast_hw_paths(n_paths, n_steps, dt, x0, gamma, xi, device, randn=None):
        k = torch.zeros((n_paths, n_steps + 1), dtype=torch.float32, device=device)
        k[:, 0] = x0
        exp_gamma_dt = np.exp(-gamma * dt)
        var_factor = float(xi * xi * (1 - np.exp(-2 * gamma * dt)) / (2 * gamma + 1e-8))
        std_factor = np.sqrt(max(var_factor, 1e-12))
        if randn is None:
            randn = torch.randn(n_paths, n_steps, device=device)
        for i in range(n_steps):
            k[:, i+1] = k[:, i] * exp_gamma_dt + std_factor * randn[:, i]
        return k
    
    def fast_simulate(n_paths, timeline, theta_vec, epsilon, v0, kappa, lam, gamma, xi,
                      f_key_vec, f_ois_vec, device, seed=None):
        n_steps = len(timeline) - 1
        dt = (timeline[1] - timeline[0]).item()
        if seed is not None:
            torch.manual_seed(seed)
        randn_v = torch.randn(n_paths, n_steps, device=device)
        randn_x = torch.randn(n_paths, n_steps, device=device)
        randn_k = torch.randn(n_paths, n_steps, device=device)
        v_paths = fast_cir_paths(n_paths, n_steps, dt, v0, kappa, theta_vec, epsilon, device, randn_v)
        x_paths = fast_ou_paths(n_paths, n_steps, dt, 0.0, lam, v_paths, device, randn_x)
        ks_paths = fast_hw_paths(n_paths, n_steps, dt, 0.0, gamma, xi, device, randn_k)
        key_paths = f_key_vec.unsqueeze(0) + x_paths + ks_paths
        ois_paths = f_ois_vec.unsqueeze(0) + x_paths
        return key_paths, ois_paths, v_paths
    
    def batch_price_caplets(key_paths, ois_paths, timeline, idx_fixes, idx_pays, strikes, tau, device):
        n_caplets = len(strikes)
        dt = (timeline[1] - timeline[0]).item()
        model_pvs = torch.zeros(n_caplets, dtype=torch.float32, device=device)
        for c in range(n_caplets):
            i_fix = idx_fixes[c].item()
            i_pay = idx_pays[c].item()
            K = strikes[c]
            if i_pay > i_fix:
                L_realized = key_paths[:, i_fix:i_pay+1].mean(dim=1)
            else:
                L_realized = key_paths[:, i_fix]
            r_integral = ois_paths[:, :i_pay+1].sum(dim=1) * dt
            disc = torch.exp(-r_integral)
            payoff = torch.clamp(L_realized - K, min=0) * tau * disc
            model_pvs[c] = payoff.mean()
        return model_pvs
    
    theta_vec_diag = theta_to_vec(theta_diag, theta_nodes_diag, timeline_device)
    
    # Simulate with many paths
    n_paths_diag = 10000
    key_paths_diag, ois_paths_diag, _ = fast_simulate(
        n_paths_diag, timeline_device, theta_vec_diag, eps_diag, v0_diag,
        kappa_diag, lam_diag, gamma_diag, xi_diag,
        f_key_vec, f_ois_vec, device, seed=999
    )
    
    # Price all caplets
    model_pvs_diag = batch_price_caplets(key_paths_diag, ois_paths_diag, timeline_device, 
                                          idx_fixes, idx_pays, strikes, tau, device)
    
    print("\nChecking all caplets for intrinsic violations...")
    
    intrinsic_violations = []
    for c in range(len(T_fixes)):
        T = T_fixes[c].item()
        K = strikes[c].item()
        F = market_fwds[c].item()
        model_pv = model_pvs_diag[c].item()
        
        # Discount factor
        T_pay = T + tau
        idx_pay = int(T_pay / dt.item())
        idx_pay = min(idx_pay, len(f_ois_vec) - 1)
        D = torch.exp(-f_ois_vec[:idx_pay+1].sum() * dt).item()
        
        intrinsic = tau * D * max(0, F - K)
        
        if model_pv < intrinsic - 1e-8:
            intrinsic_violations.append({
                'T': T, 'K': K*100, 'F': F*100, 
                'model_pv': model_pv*10000, 
                'intrinsic': intrinsic*10000,
                'shortfall': (intrinsic - model_pv)*10000
            })
    
    print(f"\nIntrinsic violations: {len(intrinsic_violations)} / {len(T_fixes)}")
    
    if intrinsic_violations:
        viol_df = pd.DataFrame(intrinsic_violations)
        viol_df = viol_df.sort_values('shortfall', ascending=False)
        print(f"\nTop 15 worst violations (model PV < intrinsic):")
        print("-"*80)
        print(viol_df.head(15).to_string(index=False, float_format=lambda x: f'{x:.2f}'))
        
        # Analyze pattern
        print(f"\n\nPattern analysis:")
        print(f"  Average T of violations: {viol_df['T'].mean():.2f}Y")
        print(f"  Average K of violations: {viol_df['K'].mean():.2f}%")
        print(f"  Average F of violations: {viol_df['F'].mean():.2f}%")
        print(f"  Average moneyness (F-K): {(viol_df['F'] - viol_df['K']).mean():.2f}%")
        print(f"  -> Violations are mostly {'ITM' if (viol_df['F'] - viol_df['K']).mean() > 0 else 'OTM'} caplets")
    else:
        print("\nâœ“ No intrinsic violations found! Model respects arbitrage bounds.")
    
    # Check overall forward bias across maturities
    print(f"\n" + "="*80)
    print("FORWARD RATE BIAS BY MATURITY")
    print("="*80)
    
    test_maturities = [1.0, 2.0, 3.0, 5.0, 7.0, 10.0]
    print(f"\n{'T(Y)':>6} {'Expected F':>12} {'Simulated F':>14} {'Bias (bp)':>12}")
    print("-"*50)
    for T_m in test_maturities:
        # Find timeline index
        idx_m = int(T_m / dt.item())
        idx_m = min(idx_m, key_paths_diag.shape[1] - 1)
        
        # Simulated forward
        sim_fwd = key_paths_diag[:, idx_m].mean().item()
        
        # Expected forward
        exp_fwd = f_key_vec[idx_m].item()
        
        bias_bp = (sim_fwd - exp_fwd) * 10000
        print(f"{T_m:6.1f} {exp_fwd*100:12.4f}% {sim_fwd*100:14.4f}% {bias_bp:12.2f}")
    
    print("\n" + "="*80)

INTRINSIC VALUE DIAGNOSTIC
Checkpoint keys: ['iteration', 'theta_values', 'epsilon', 'time_to_maturities', 'n_maturities', 'loss', 'v0', 'kappa', 'lam', 'gamma', 'xi', 'k0', 'history']
Theta nodes: [ 0.08333334  0.16666667  0.25        0.5         0.75        1.
  2.          3.          4.          5.          6.          7.
  8.          9.         10.        ]
Theta values: [0.00252559 0.00442503 0.00350415 0.00364944 0.00218418 0.00253815
 0.00342201 0.00319243 0.00387829 0.00274906 0.00198515 0.00148707
 0.00064807 0.00152052 0.00431108]
Epsilon: 0.49926123752488105, v0: 0.00019999999494757503


C:\Users\Andrey\AppData\Local\Temp\ipykernel_6668\3373745295.py:53: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  exp_lam_dt = np.exp(-lam * dt)
C:\Users\Andrey\AppData\Local\Temp\ipykernel_6668\3373745295.py:63: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  exp_gamma_dt = np.exp(-gamma * dt)
C:\Users\Andrey\AppData\Local\Temp\ipykernel_6668\3373745295.py:64: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  var_factor = float(xi * xi * (1 - np.exp(-2 * gamma * dt)) / (2 * gamma + 1e-8))



Checking all caplets for intrinsic violations...

Intrinsic violations: 12 / 495

Top 15 worst violations (model PV < intrinsic):
--------------------------------------------------------------------------------
   T    K     F  model_pv  intrinsic  shortfall
0.50 3.00 16.33    289.50     295.52       6.01
0.50 4.00 16.33    268.00     273.35       5.35
0.50 5.00 16.33    246.61     251.18       4.57
0.50 6.00 16.33    225.38     229.01       3.63
0.75 3.00 15.67    266.85     270.26       3.41
0.50 6.50 16.33    214.82     217.93       3.10
0.50 7.00 16.33    204.33     206.84       2.51
0.75 4.00 15.67    246.79     248.93       2.14
0.50 7.50 16.33    193.90     195.76       1.86
0.50 8.00 16.33    183.54     184.67       1.13
0.75 5.00 15.67    226.93     227.60       0.67
0.50 8.50 16.33    173.27     173.59       0.32


Pattern analysis:
  Average T of violations: 0.56Y
  Average K of violations: 5.62%
  Average F of violations: 16.16%
  Average moneyness (F-K): 10.54%
  -> Viola